In [1]:
!pip install -q llama-index pymilvus transformers sentence-transformers torch

In [2]:
import os
import sys
from pathlib import Path
import gc

# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))

import re
import unicodedata
import torch
import pickle
import random
from typing import List, Optional, Tuple
import json
import time
from datetime import datetime, timedelta
import psutil
import numpy as np
from sentence_transformers import SentenceTransformer
from pymilvus import (
    FieldSchema, CollectionSchema, DataType, Collection, Index, 
    MilvusClient, utility, connections
)  # :contentReference[oaicite:3]{index=3}
from IPython.core.interactiveshell import InteractiveShell
from src.utils.data_utils import load_documents, select_random_documents
from src.utils.evaluation import (
    perform_analysis, 
    perform_single_match_analysis,
    print_single_match_analysis,
    print_sample_results
)

# Adjust Jupyter Notebook settings to avoid trucating long strings
InteractiveShell.ast_node_interactivity = "all" 

# Check if CUDA is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def sanitize_collection_name(name: str) -> str:
    # Normalize string to NFKD (decomposes accents)
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if unicodedata.category(c) != 'Mn')  # Remove diacritics
    
    # Replace invalid characters with underscores (only allow letters, numbers, and underscores)
    name = re.sub(r'[^a-zA-Z0-9_]', '_', name)
    
    return name

def create_index(model_name, documents_path, collection_name, batch_size=2000):
    
    total_start_time = time.time()
    
    # Initialize the transformer embedding model
    embed_model = SentenceTransformer(model_name, device=str(device))
    embedding_dim = embed_model.get_sentence_embedding_dimension()
    print(f"➡️ Model {model_name} loaded with embedding dimension: {embedding_dim}")

    # Load your documents (assumed to be a pickled list of objects/dicts with 'text' and optionally 'item_key')
    try:
        with open(documents_path, 'rb') as file:
            documents = pickle.load(file)
        print(f"Successfully loaded {len(documents)} documents from {documents_path}")
    except Exception as e:
        print(f"An error occurred while loading the documents: {str(e)}")
        return None
   
    # Prepare lists for texts and optional item keys
    texts = []
    item_keys = []
    for doc in documents:
        # Extract the item key from metadata
        key = doc.metadata.get('item_key', '')
        # Extract the text from the text_resource attribute
        txt = doc.text_resource.text if doc.text_resource is not None else ''
        texts.append(txt)
        item_keys.append(key)
        
    total_docs = len(texts)
    print(f"Total documents to process: {total_docs}")

    # Compute embeddings in mini-batches to manage GPU memory
    all_embeddings = []
    mini_batch_size = 2048
    for i in range(0, total_docs, mini_batch_size):
        mini_texts = texts[i:i+mini_batch_size]
        mini_embeddings = embed_model.encode(mini_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        all_embeddings.extend(mini_embeddings)
        if device.type == "cuda":
            torch.cuda.empty_cache()
    print("Completed computing embeddings for all documents.")

    # Connect to Milvus
    # Here we use MilvusClient (ensure your Milvus server is running at host.docker.internal:19530)
    # Ensure connection using `connections.connect()`
    connections.connect(alias="default", host="host.docker.internal", port=19530)
    client = MilvusClient(uri="http://host.docker.internal:19530")
    print(f"Milvus Connection Established:")

    # If the collection already exists, drop it (or you may choose to update instead)
    if client.has_collection(collection_name):
        print(f"Collection '{collection_name}' already exists. Dropping the collection.")
        utility.drop_collection(collection_name)

    # Define the collection schema: auto-generated primary key, a VARCHAR field for item_key and text, and a FLOAT_VECTOR for embeddings.
    fields = [
        FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
        FieldSchema(name="item_key", dtype=DataType.VARCHAR, max_length=100, description="Document item key"),
        FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535, description="Document text"),
        FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=embedding_dim, description="Embedding vector")
    ]
    schema = CollectionSchema(fields, description="Text embedding collection")
    collection = Collection(name=collection_name, schema=schema)
    print(f"Created Milvus collection '{collection_name}'.")

    # Insert the documents into Milvus in batches
    total_batches = (total_docs + batch_size - 1) // batch_size
    for batch_idx in range(total_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, total_docs)
        batch_item_keys = item_keys[start_idx:end_idx]
        batch_texts = texts[start_idx:end_idx]
        batch_vectors = all_embeddings[start_idx:end_idx]
        
        # Arrange the data for each field; note that the 'id' field is auto-generated
        data = [
            batch_item_keys,
            batch_texts,
            batch_vectors
        ]
        insert_result = collection.insert(data)
        print(f"Inserted batch {batch_idx+1}/{total_batches} with {end_idx - start_idx} records.")

        # Create an index
        index_params = {
            "metric_type": "IP",  
            "index_type": "IVF_FLAT",  
            "params": {"nlist": int(total_docs/100)}  
        }

        # Create the index on the vector field
        index = Index(collection, "vector", index_params)
        print("Index created successfully!")
        
        collection.flush()
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        
    total_time = time.time() - total_start_time
    print(f"Total index creation time: {timedelta(seconds=int(total_time))}")
    print("-"*50)
    print(f"/n")
    # Return both the Milvus collection and the embedding model handle for retrieval
    return {"collection": collection, "embed_model": embed_model}

def save_to_file(content, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(content)

def main(file):
    """
    Main pipeline function.
    
    Parameters:
    - file
    """
    start_time = time.time()
    
    model_options = {
        "hiiamsid": "hiiamsid/sentence_similarity_spanish_es",
        "somosnlp": "somosnlp-hackathon-2022/paraphrase-spanish-distilroberta",
        "mrm8488": "mrm8488/multilingual-e5-large-ft-sts-spanish-matryoshka-768-16-5e"
    }

    documents_path_texto = Path('/work/data/processed/' + file + '_texto.pkl')
    collection_name_texto = file.replace(" ","_") + '_texto_collection' # Milvus does not allow spaces in collection names
    collection_name_texto = sanitize_collection_name(collection_name_texto)
    
    documents_path_resumen = Path('/work/data/processed/' + file + '_resumen.pkl')
    collection_name_resumen = file.replace(" ","_") + '_resumen_collection'
    collection_name_resumen = sanitize_collection_name(collection_name_resumen)

    # Create output directory
    output_dir = Path('Output')
    output_dir.mkdir(exist_ok=True)
    
    # Choose your model
    chosen_model = model_options["hiiamsid"]

    # Create indexes using the new Milvus-based method
    index_texto = create_index(chosen_model, documents_path_texto, collection_name_texto, batch_size=2000)
    index_resumen = create_index(chosen_model, documents_path_resumen, collection_name_resumen, batch_size=2000)
    
    if index_resumen is None or index_texto is None:
        print("Failed to create one or both indexes. Exiting.")
        return

    # Save index information
    index_info = {
        "texto_collection": collection_name_texto,
        "resumen_collection": collection_name_resumen,
        "model_used": chosen_model
    }
    save_to_file(json.dumps(index_info, indent=2), output_dir / 'index_info.json')
    
    # Load documents
    documents_texto = load_documents(documents_path_texto)
    documents_resumen = load_documents(documents_path_resumen)

    if documents_texto is None or documents_resumen is None:
        print("Failed to load one or both document sets. Exiting.")
        return


Using device: cuda


In [ ]:
main('OEB')

➡️ Model hiiamsid/sentence_similarity_spanish_es loaded with embedding dimension: 768
Successfully loaded 47514 documents from /work/data/processed/OEB_texto.pkl
Total documents to process: 47514
Completed computing embeddings for all documents.
Milvus Connection Established:
Created Milvus collection 'OEB_texto_collection'.
Inserted batch 1/24 with 2000 records.
Index created successfully!
Inserted batch 2/24 with 2000 records.
Index created successfully!
Inserted batch 3/24 with 2000 records.
Index created successfully!
Inserted batch 4/24 with 2000 records.
Index created successfully!
Inserted batch 5/24 with 2000 records.
Index created successfully!
Inserted batch 6/24 with 2000 records.
Index created successfully!
